In [1]:
source("../helpers/map.r")
source("../helpers/setup.r")
source("../helpers/clinical_help.r")

Warning message in file(con, "r"):
“cannot open file '/data/tmp/vignettes-base/time_stamp.txt': No such file or directory”


# 0 - Queries

In [2]:
clinicalSummary   <- runQuery('clinical_summary')
treatment         <- runQuery('treatment')
treatmentResponse <- runQuery('treatmentResponse')
patient           <- runQuery('patient') %>% filter(blacklisted == 0) %>% select(-blacklisted)
sample            <- runQuery('sample') %>% filter(type == "DNA") %>% select(-type)

# 1 - Prepare metadata

In [3]:
metaReady <-
  clinicalSummary %>%
  mutate(across(where(is.character), ~ na_if(., "null")),
         vitalStatus = ifelse(vitalStatus %in% c("Alive", "Deceased"), vitalStatus, NA)) %>%
  rowwise() %>%
  mutate(age = age(birthYear, biopsyDate)) %>%
  ungroup() %>%
  transmute(
    sampleId, specimenId, donorId, purpleQC, isofoxQC, doids, purity = tumorPurity, 
    age, sex, primaryTumorLocation, primaryTumorType,
    biopsyId, biopsyDate, biopsyStructure, biopsyType,
    hasSystemicPreTreatment, hasRadiotherapyPreTreatment,
    vitalStatus, dateLastFollowUp)

# 2 - Prepare treatment

In [4]:
treatmentGo <- 
patient %>% 
  transmute(patientId = id, donorId) %>%
  inner_join(sample, by = "donorId") %>% 
  inner_join(treatment %>% rename(treatmentId = id), by = c("patientId", "biopsyId")) %>%
  select(-patientId)

# 3 - Prepare responses

## 3.1 - Map RECIST responses

In [5]:
response <-
  treatmentResponse %>%
  filter(response != "STOP_TREATMENT;DEATH", treatmentId != "NULL") %>%
  mutate(
    responseRecist = unname(response_map[response]), 
    treatmentId = as.numeric(treatmentId),
    response    = ifelse(responseRecist == "ND", NA, responseRecist %in% c("CR", "PR")),
    benefit     = ifelse(responseRecist == "ND", NA, responseRecist %in% c("CR", "PR", "SD")),
    progression = ifelse(responseRecist == "ND", NA, responseRecist == "PD")
  )

In [6]:
unmapped <- treatmentResponse %>%
  filter(response != "STOP_TREATMENT;DEATH", treatmentId != "NULL") %>%
  filter(!response %in% names(response_map)) %>%
  pull(response) %>%
  unique()
if (length(unmapped) > 0) warning("Unmapped response values: ", paste(unmapped, collapse = ", "))

# 3.2 Combine with Treatment

In [7]:
treatmentResponseGo <- treatmentGo %>% inner_join(response, by = "treatmentId")

# 4 - Derive outcomes

In [8]:
longDataReady <-
  treatmentResponseGo %>%
  left_join(metaReady %>% transmute(sampleId, vitalStatus, dateLastFollowUp), 
            by = "sampleId") %>%
  rowwise() %>%
  mutate(
    daysToTreatmentResponse    = round(date_diff(startDate, responseDate)),
    effectiveDateLastFollowUp = max_date(safe_date(endDate), safe_date(dateLastFollowUp)),
    daysToLastMeasurement = round(date_diff(startDate, effectiveDateLastFollowUp)),
    daysToDeath = ifelse(vitalStatus == "Deceased", daysToLastMeasurement, NA), 
    daysToProgression          = ifelse(progression, daysToTreatmentResponse, NA),
    daysToBenefit              = ifelse(benefit,     daysToTreatmentResponse, NA)) %>%
  ungroup() 

In [9]:
check <- 
longDataReady %>%
 select(sampleId, vitalStatus, daysToLastMeasurement) %>%
 unique()

In [10]:
sampleOutcomes <-
  longDataReady %>%
  group_by(sampleId, name, startDate, endDate, type, mechanism) %>%
  summarise(
    bestOverallResponse        = as.numeric(any(response, na.rm = TRUE)),
    maxDaysToBenefit           = clean(daysToBenefit,    FUN = max),
    minDaysToProgression       = clean(daysToProgression, FUN = min),
    daysToLastAliveMeasurement = first(daysToLastMeasurement),
    daysToDeath                = first(daysToDeath),
    responses                  = paste0(unique(responseRecist), collapse = ","),
    .groups = 'drop') %>%
  rowwise() %>%
  mutate(
    durableClinicalBenefit             = dcb(maxDaysToBenefit, minDaysToProgression, days = 182, method = "strict"),
    durableClinicalBenefitRelaxed      = dcb(maxDaysToBenefit, minDaysToProgression, days = 182, method = "relaxed"),
    durableClinicalBenefitShort        = dcb(maxDaysToBenefit, minDaysToProgression, days = 91,  method = "strict"),
    durableClinicalBenefitShortRelaxed = dcb(maxDaysToBenefit, minDaysToProgression, days = 91,  method = "relaxed"),
    pfsEvent      = as.numeric(!is.na(minDaysToProgression) | !is.na(daysToDeath)),
    pfsEventDays  = pfs_days(minDaysToProgression, daysToLastAliveMeasurement , daysToDeath),
    osEvent       = as.numeric(!is.na(daysToDeath)),
    osDays        = os_days(daysToLastAliveMeasurement, daysToDeath),
    response      = as.numeric(bestOverallResponse + durableClinicalBenefit > 0),
    responseShort = as.numeric(bestOverallResponse + durableClinicalBenefitShort > 0),
    recistBest   = best_recist(responses),
    bestOverallResponse = ifelse(responses == "ND", NA, bestOverallResponse)) %>%
  ungroup() %>%
  transmute(
    sampleId,
    treatmentStartDate = startDate, treatmentEndDate = endDate,
    treatment = name, type, mechanism,
    bestOverallResponse, recistBest,
    durableClinicalBenefit, durableClinicalBenefitRelaxed,
    durableClinicalBenefitShort, durableClinicalBenefitShortRelaxed,
    response, responseShort,
    pfsEvent, pfsEventDays, osEvent, osDays)

# 5 - Clinical ready

In [11]:
clinicalReady <-
  metaReady %>%
  left_join(sampleOutcomes, by = "sampleId") %>%
  rowwise() %>%
  mutate(timeBiopsyToTreatmentDays = date_diff(biopsyDate, treatmentStartDate)) %>%
  ungroup() %>%
  mutate(osEvent = ifelse(is.na(vitalStatus), NA, osEvent))

# 6 - Save

In [19]:
fwrite(clinicalReady %>% mutate(pipeline_date = PIPELINE_DATE), 
       file = paste0(BASE_DATA, "clinical.txt"))